In [90]:
# Imports
import numpy as np
import torch 
import torch.nn as nn
import soundfile as sf
from torch.utils.data import TensorDataset, DataLoader, random_split

In [91]:
# Read the two audi files
dry_audio, sr = sf.read("dataset_dry.wav")
wet_audio, sr = sf.read("dataset_wet.wav")

In [92]:
# Convert to PyTorch tensors
dry_tensor = torch.tensor(dry_audio, dtype=torch.float32)
wet_tensor = torch.tensor(wet_audio, dtype=torch.float32)

dry_tensor = dry_tensor.unsqueeze(1)  # Add batch dimension
wet_tensor = wet_tensor.unsqueeze(1)  # Add batch dimension

# Get the shape of the tensor
dry_shape = dry_tensor.shape
wet_shape = wet_tensor.shape

print('Tensor: ', dry_tensor)
print('Tensor shape: ', dry_shape)
print('Tensor: ', wet_tensor)
print('Tensor shape: ', wet_shape)

Tensor:  tensor([[0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        ...,
        [3.0518e-05],
        [9.1553e-05],
        [1.5259e-04]])
Tensor shape:  torch.Size([3423232, 1])
Tensor:  tensor([[0.0000],
        [0.0000],
        [0.0000],
        ...,
        [0.0016],
        [0.0030],
        [0.0048]])
Tensor shape:  torch.Size([3423232, 1])


### Building the Neural Network (Neural Amp Model)
Now that our data is formatted as PyTorch Tensors, it's time to build the "brain" of our digital amplifier. 
We will use an architecture with an **LSTM** layer (to help the model "remember" how the sound wave evolves over time) and a **Linear** layer to produce the final distorted audio signal.

In [93]:
class GuitarAmpModel(nn.Module):
    def __init__(self):
        super().__init__() 
        
        # Create the layers of our model
        
        # 1. Add an LSTM layer that takes 1 input and outputs 32 hidden units
        self.lstm = nn.LSTM(input_size=1, hidden_size=32, batch_first=True)
        
        # 2. Add a Linear layer that takes 32 inputs and outputs 1
        self.linear = nn.Linear(in_features=32, out_features=1)

    def forward(self, x):
        # Here we will pass the data through the layers of our model
        
        x = self.lstm(x)[0]  # Pass through LSTM and take the output
        x = self.linear(x)    # Pass through Linear layer
        return x
    
# Transfer the model to GPU if available, otherwise to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GuitarAmpModel().to(device)
print(model)

GuitarAmpModel(
  (lstm): LSTM(1, 32, batch_first=True)
  (linear): Linear(in_features=32, out_features=1, bias=True)
)


In [94]:
# Parameters
num_epochs = 50
learning_rate = 0.001
batch_size = 64 

In [95]:
# Create an instance of the model
model = GuitarAmpModel()

# Define a loss function and an optimizer
criterion = nn.MSELoss()  # Mean Squared Error loss
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  # Adam optimizer

In [96]:
# Prepare the data for training
chunk_size = 4096
num_chunks = len(dry_tensor) // chunk_size

# Reshape the tensors into chunks
dry_chunks = dry_tensor[:num_chunks * chunk_size].view(num_chunks, chunk_size, 1)
wet_chunks = wet_tensor[:num_chunks * chunk_size].view(num_chunks, chunk_size, 1)

print(f"New shape: {dry_chunks.shape}")
print(f"New shape: {wet_chunks.shape}")

New shape: torch.Size([835, 4096, 1])
New shape: torch.Size([835, 4096, 1])


In [97]:
def train(model, dataloader, optimizer, criterion, device):
    # Set the model to training mode
    model.train()

    total_loss = 0.0

    # Iterate over training data
    for batch, (inputs, targets) in enumerate(dataloader):
        inputs = inputs.to(device)
        targets = targets.to(device)

        # Forward pass: prediction
        outputs = model(inputs) 
        
        # Calculate the loss between the predicted outputs and the actual targets
        loss = criterion(outputs, targets) 
        total_loss += loss.item()

        # Backpropagation & optimization
        optimizer.zero_grad() 
        loss.backward() 
        optimizer.step() 

        # Print progress
        if batch % 100 == 0:
            current = batch * dataloader.batch_size + len(inputs)
            print(f"loss: {loss.item():>7f}  [{current:>5d}/{len(dataloader.dataset):>5d}]")

    # Return average loss for this epoch
    return total_loss / len(dataloader)

In [98]:
def test(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()

            all_preds.extend(outputs.detach().cpu().numpy())
            all_targets.extend(targets.detach().cpu().numpy())

    return total_loss / len(dataloader), np.array(all_preds), np.array(all_targets)

In [99]:
def run_training(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    history = {'train_loss': [], 'val_loss': []}
    print(f"Starting training on {device} for {epochs} epochs...")

    for epoch in range(epochs):
        t_loss = train(model, train_loader, optimizer, criterion, device) 
        v_loss, _, _ = test(model, val_loader, criterion, device)

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f}")

    return history

In [100]:
# Create a TensorDataset and DataLoader
full_dataset = TensorDataset(dry_chunks, wet_chunks)

# Define the sizes for the split (e.g., 80% train, 20% val)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Create DataLoaders for training and validation
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 11
Val batches: 3


In [101]:
history = run_training(model, train_loader, val_loader, optimizer, criterion, device, num_epochs)

Starting training on cpu for 50 epochs...
loss: 0.262954  [   64/  668]
Epoch 1/50 | Train Loss: 0.2418 | Val Loss: 0.2126
loss: 0.226940  [   64/  668]
Epoch 2/50 | Train Loss: 0.2460 | Val Loss: 0.2111
loss: 0.253375  [   64/  668]
Epoch 3/50 | Train Loss: 0.2374 | Val Loss: 0.2093
loss: 0.263441  [   64/  668]
Epoch 4/50 | Train Loss: 0.2362 | Val Loss: 0.2058
loss: 0.246787  [   64/  668]
Epoch 5/50 | Train Loss: 0.2276 | Val Loss: 0.1939
loss: 0.210608  [   64/  668]
Epoch 6/50 | Train Loss: 0.2074 | Val Loss: 0.1768
loss: 0.206902  [   64/  668]
Epoch 7/50 | Train Loss: 0.1889 | Val Loss: 0.1496
loss: 0.155952  [   64/  668]
Epoch 8/50 | Train Loss: 0.1333 | Val Loss: 0.0980
loss: 0.118879  [   64/  668]
Epoch 9/50 | Train Loss: 0.1016 | Val Loss: 0.0820
loss: 0.092086  [   64/  668]
Epoch 10/50 | Train Loss: 0.0854 | Val Loss: 0.0719
loss: 0.072026  [   64/  668]
Epoch 11/50 | Train Loss: 0.0727 | Val Loss: 0.0628
loss: 0.071647  [   64/  668]
Epoch 12/50 | Train Loss: 0.0612 | 

In [102]:
# Evaluate the model on the validation set and get predictions
loss, predictions, targets = test(model, val_loader, criterion, device)

# Convert predictions to a continuous sequence of samples
predicted_audio = predictions.flatten()

sf.write('./notebook_tests/model_output.wav', predicted_audio, sr)

### Full Audio Inference 🎸
The `model_output.wav` file we just saved only contains short validation chunks. 
To use our model practically, we need a function that takes a complete, dry audio track, automatically splits it into chunks, applies the AI distortion, and smoothly concatenates the pieces back together into a single, continuous track!

In [104]:
def predict_full_audio(model, dry_tensor, chunk_size, device):
    model.eval()
    output_chunks = []
    
    # Split the audio into chunks
    num_chunks = len(dry_tensor) // chunk_size
    
    with torch.no_grad():
        for i in range(num_chunks):
            # Take the current chunk
            start = i * chunk_size
            end = start + chunk_size
            chunk = dry_tensor[start:end].view(1, chunk_size, 1).to(device)
            
            # Predict from the model
            prediction = model(chunk)
            
            # Store the output (move to CPU and convert to numpy)
            output_chunks.append(prediction.cpu().numpy().flatten())
            
    # Concatenate all chunks into a single array
    return np.concatenate(output_chunks)

In [105]:
# Make a prediction for the full audio
predicted_audio = predict_full_audio(model, dry_tensor, chunk_size, device)

# Write the .wav file
sf.write('./notebook_tests/predicted_full_audio.wav', predicted_audio, sr)